# フィールドAPI: 国土地理院 DEM から空間スペクトルへ

自分で取得した GSI FGD DEM GML/ZIP ファイルのパスを `GWEXPY_DEM_TILES` に区切り文字付きで指定します。未指定時は合成地理ラスタで実行します。読み込み、欠損値の明示的な補間、局所座標変換、空間 FFT を順に示します。

## 準備

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from gwexpy.fields import GeoMetadata, ScalarField

## 地形の読み込みまたは生成

In [ ]:
paths = [Path(p) for p in os.environ.get("GWEXPY_DEM_TILES", "").split(os.pathsep) if p]
if paths:
    missing = [str(p) for p in paths if not p.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    dem = ScalarField.read(paths, "elevation", format="gsi-dem")
else:
    # Offline geographic raster; rows descend from north to south.
    lon = np.linspace(137.30, 137.32, 64) * u.deg
    lat = np.linspace(36.42, 36.40, 48) * u.deg
    xx, yy = np.meshgrid(lon.value, lat.value, indexing="ij")
    elevation = 400 + 180 * np.exp(-((xx - 137.31) / 0.004)**2 - ((yy - 36.41) / 0.005)**2)
    elevation[20:24, 15:19] = np.nan
    dem = ScalarField(elevation[None, :, :, None], unit=u.m, name="elevation",
        axis0=[0.0] * u.s, axis1=lon, axis2=lat, axis3=[0.0] * u.m,
        axis_names=("t", "lon", "lat", "z"),
        space_domain={"lon": "angle", "lat": "angle", "z": "real"})
    dem.geo = GeoMetadata(crs="EPSG:6668", source_format="synthetic", active=True)
print(dem.shape, dem.axis_names, dem.geo)

## 補間・座標変換・FFT

In [ ]:
# NoData remains NaN after read; infilling is explicit.
filled = dem.fill_missing(method="nearest")
assert np.isfinite(filled.value).all()
local = filled.to_local_cartesian()
assert local.space_domains["x"] == "real"
# Forward scipy.fft.fftn is unnormalized; the output axes are angular wavenumber.
spectrum = local.fft_space(["x", "y"], spacing_rtol=1e-8)
power = np.abs(spectrum.value[0, :, :, 0])**2
print(spectrum.shape, spectrum.axis_names, spectrum.geo.history[-1])

## 地形とスペクトルを確認

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(local.value[0, :, :, 0].T, origin="lower", aspect="auto")
axes[0].set_title("Elevation in local frame [m]")
axes[1].imshow(np.log1p(np.fft.fftshift(power)).T, origin="lower", aspect="auto")
axes[1].set_title("log(1 + spatial spectral power)")
for ax in axes:
    ax.set_xlabel("grid x index")
    ax.set_ylabel("grid y index")
fig.tight_layout()
plt.show()

局所接平面への変換は狭い領域での近似です。広域には optional extra `geospatial` と `to_projected()` を使います。`spacing_rtol` はほぼ等間隔の標本に対する明示的な許容誤差です。順 FFT は非正規化で、座標は角波数です。